In [ ]:
import pandas as pd
import numpy as np
import os

# Cargamos el dataset (ajusta el nombre exacto de tu archivo CSV)
path = "../data/raw/MUNIFI Data Query - DISAGGREGATED DATA_Chile.csv" 
df = pd.read_csv(path)

print("¡Dataset cargado con éxito!")

¡Dataset cargado con éxito!


In [2]:
display(df.head())
print("-" * 30)
df.info()

,Country,ISCO3,TL 2 Name,TL 2 Code,TL 3 Name,TL 3 Code,Municipality,REF_AREA,TIME_PERIOD,MEASURE,Measure name,UNIT_MEASURE,Unit of measure,OBS_VALUE
0,Chile,CHL,Antofagasta,CL02,Antofagasta,CL021,ANTOFAGASTA,CHL02101,2010.0,E1,Expenditure,XDC,National currency,3.390620e+10
1,Chile,CHL,Antofagasta,CL02,Antofagasta,CL021,ANTOFAGASTA,CHL02101,2011.0,E1,Expenditure,XDC,National currency,3.943266e+10
2,Chile,CHL,Antofagasta,CL02,Antofagasta,CL021,ANTOFAGASTA,CHL02101,2012.0,E1,Expenditure,XDC,National currency,5.359428e+10
3,Chile,CHL,Antofagasta,CL02,Antofagasta,CL021,ANTOFAGASTA,CHL02101,2013.0,E1,Expenditure,XDC,National currency,6.746200e+10
4,Chile,CHL,Antofagasta,CL02,Antofagasta,CL021,ANTOFAGASTA,CHL02101,2014.0,E1,Expenditure,XDC,National currency,6.591384e+10


------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351900 entries, 0 to 351899
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Country          351900 non-null  object 
 1   ISCO3            351900 non-null  object 
 2   TL 2 Name        351900 non-null  object 
 3   TL 2 Code        351900 non-null  object 
 4   TL 3 Name        351900 non-null  object 
 5   TL 3 Code        351900 non-null  object 
 6   Municipality     351900 non-null  object 
 7   REF_AREA         351900 non-null  object 
 8   TIME_PERIOD      351900 non-null  float64
 9   MEASURE          351900 non-null  object 
 10  Measure name     351900 non-null  object 
 11  UNIT_MEASURE     351900 non-null  object 
 12  Unit of measure  351900 non-null  object 
 13  OBS_VALUE        269100 non-null  float64
dtypes: float64(2), object(12)
memory usage: 37.6+ MB


In [3]:
# Chequeo de nulos
null_count = df.isnull().sum()
null_percentage = (null_count / len(df)) * 100

missing_data = pd.DataFrame({
    'Valores Nulos': null_count,
    'Porcentaje (%)': null_percentage
}).sort_values(by='Porcentaje (%)', ascending=False)

print("Análisis de datos faltantes:")
display(missing_data)

print(f"\nCantidad de filas duplicadas: {df.duplicated().sum()}")

Análisis de datos faltantes:


,Valores Nulos,Porcentaje (%)
OBS_VALUE,82800,23.529412
Country,0,0.000000
ISCO3,0,0.000000
TL 2 Name,0,0.000000
TL 2 Code,0,0.000000
TL 3 Name,0,0.000000
TL 3 Code,0,0.000000
Municipality,0,0.000000
REF_AREA,0,0.000000
TIME_PERIOD,0,0.000000



Cantidad de filas duplicadas: 0


In [4]:
key_cols = ['Country','TL 2 Name','TL 3 Name','Municipality','TIME_PERIOD','MEASURE']

dup_mask = df.duplicated(subset=key_cols, keep=False)
dup_count = int(dup_mask.sum())
print(f"Filas con duplicado lógico según llave {key_cols}: {dup_count}")

if dup_count > 0:
    display(df.loc[dup_mask, key_cols + ['OBS_VALUE']].sort_values(key_cols).head(20))
    dup_summary = (
        df.loc[dup_mask]
          .groupby('MEASURE')
          .size()
          .sort_values(ascending=False)
          .to_frame('dup_rows')
    )
    display(dup_summary)
else:
    print('OK: no hay duplicados lógicos para municipio-año-medida con la llave definida.')

Filas con duplicado lógico según llave ['Country', 'TL 2 Name', 'TL 3 Name', 'Municipality', 'TIME_PERIOD', 'MEASURE']: 351900


,Country,TL 2 Name,TL 3 Name,Municipality,TIME_PERIOD,MEASURE,OBS_VALUE
0,Chile,Antofagasta,Antofagasta,ANTOFAGASTA,2010.0,E1,3.390620e+10
4140,Chile,Antofagasta,Antofagasta,ANTOFAGASTA,2010.0,E1,9.238267e+04
8280,Chile,Antofagasta,Antofagasta,ANTOFAGASTA,2010.0,E1,9.605784e+07
12420,Chile,Antofagasta,Antofagasta,ANTOFAGASTA,2010.0,E1,2.617244e+02
16560,Chile,Antofagasta,Antofagasta,ANTOFAGASTA,2010.0,E1,NaN
20700,Chile,Antofagasta,Antofagasta,ANTOFAGASTA,2010.0,E11,3.203128e+10
24840,Chile,Antofagasta,Antofagasta,ANTOFAGASTA,2010.0,E11,8.727418e+04
28980,Chile,Antofagasta,Antofagasta,ANTOFAGASTA,2010.0,E11,9.074612e+07
33120,Chile,Antofagasta,Antofagasta,ANTOFAGASTA,2010.0,E11,2.472518e+02
37260,Chile,Antofagasta,Antofagasta,ANTOFAGASTA,2010.0,E11,NaN


,dup_rows
MEASURE,
E1,20700
EC8,20700
R13,20700
R12,20700
R115,20700
R11,20700
R1,20700
POP,20700
EC10,20700


In [5]:
key_cols = ['Country','TL 2 Name','TL 3 Name','Municipality','TIME_PERIOD','MEASURE']

# 1) ¿Cuántas llaves únicas hay vs filas totales?
print("Filas totales:", len(df))
print("Llaves únicas:", df[key_cols].drop_duplicates().shape[0])

# 2) ¿Cuál es la máxima cantidad de repeticiones de una misma llave?
vc = df.groupby(key_cols).size().sort_values(ascending=False)
print("Máxima repetición de una llave:", int(vc.iloc[0]))
display(vc.head(10))

# 3) Ver un ejemplo concreto de una llave repetida y comparar columnas
example_key = vc.index[0]
mask = (df[key_cols] == pd.Series(example_key, index=key_cols)).all(axis=1)
display(df.loc[mask].head(20))

Filas totales: 351900
Llaves únicas: 70380
Máxima repetición de una llave: 5


Country  TL 2 Name              TL 3 Name    Municipality  TIME_PERIOD  MEASURE
Chile    Antofagasta            Antofagasta  ANTOFAGASTA   2010.0       E1         5
         Santiago Metropolitan  Chacabuco    TILTIL        2021.0       R12        5
                                Cordillera   PIRQUE        2010.0       E115       5
                                                                        E111       5
                                                                        E11        5
                                                                        E1         5
                                Chacabuco    TILTIL        2021.0       R14        5
                                                                        R13        5
                                                                        R115       5
                                Cordillera   PIRQUE        2011.0       E121       5
dtype: int64

,Country,ISCO3,TL 2 Name,TL 2 Code,TL 3 Name,TL 3 Code,Municipality,REF_AREA,TIME_PERIOD,MEASURE,Measure name,UNIT_MEASURE,Unit of measure,OBS_VALUE
0,Chile,CHL,Antofagasta,CL02,Antofagasta,CL021,ANTOFAGASTA,CHL02101,2010.0,E1,Expenditure,XDC,National currency,3.390620e+10
4140,Chile,CHL,Antofagasta,CL02,Antofagasta,CL021,ANTOFAGASTA,CHL02101,2010.0,E1,Expenditure,XDC_PS,National currency per person,9.238267e+04
8280,Chile,CHL,Antofagasta,CL02,Antofagasta,CL021,ANTOFAGASTA,CHL02101,2010.0,E1,Expenditure,USD_PPP,"US dollars, PPP converted",9.605784e+07
12420,Chile,CHL,Antofagasta,CL02,Antofagasta,CL021,ANTOFAGASTA,CHL02101,2010.0,E1,Expenditure,USD_PPP_PS,"US dollars, PPP converted per person",2.617244e+02
16560,Chile,CHL,Antofagasta,CL02,Antofagasta,CL021,ANTOFAGASTA,CHL02101,2010.0,E1,Expenditure,PS,Persons,NaN


In [6]:
key_cols = ['Country','TL 2 Name','TL 3 Name','Municipality','TIME_PERIOD','MEASURE']

vc = df.groupby(key_cols).size().sort_values(ascending=False)
example_key = vc.index[0]

mask = (df[key_cols] == pd.Series(example_key, index=key_cols)).all(axis=1)
rows = df.loc[mask].copy()

print("Ejemplo de llave:", example_key)
print("Cantidad de filas para esa llave:", len(rows))

# Mostrar columnas donde hay más de un valor distinto (o sea: las que explican la repetición)
varying = [c for c in rows.columns if rows[c].nunique(dropna=False) > 1]
print("Columnas que cambian entre las 5 filas:", varying)

display(rows[varying + ['OBS_VALUE']].head(10))

Ejemplo de llave: ('Chile', 'Antofagasta', ' Antofagasta', 'ANTOFAGASTA', 2010.0, 'E1')
Cantidad de filas para esa llave: 5
Columnas que cambian entre las 5 filas: ['UNIT_MEASURE', 'Unit of measure', 'OBS_VALUE']


,UNIT_MEASURE,Unit of measure,OBS_VALUE,OBS_VALUE
0,XDC,National currency,3.390620e+10,3.390620e+10
4140,XDC_PS,National currency per person,9.238267e+04,9.238267e+04
8280,USD_PPP,"US dollars, PPP converted",9.605784e+07,9.605784e+07
12420,USD_PPP_PS,"US dollars, PPP converted per person",2.617244e+02,2.617244e+02
16560,PS,Persons,NaN,NaN


In [7]:
unit_targets = ['XDC', 'XDC_PS', 'USD_PPP', 'USD_PPP_PS']
base_index_cols = ['Country','TL 2 Name','TL 3 Name','Municipality','REF_AREA','TIME_PERIOD']

panels = {}
long_by_unit = {}

for u in unit_targets:
    df_u = df[df['UNIT_MEASURE'] == u].copy()
    df_u['TIME_PERIOD'] = df_u['TIME_PERIOD'].astype(int)

    long_by_unit[u] = df_u

    panel_u = (
        df_u.pivot_table(
            index=base_index_cols,
            columns='MEASURE',
            values='OBS_VALUE',
            aggfunc='first'
        )
        .reset_index()
    )

    panels[u] = panel_u

    print(f"UNIT_MEASURE={u}")
    print(f"  Long shape:  {df_u.shape}")
    print(f"  Panel shape: {panel_u.shape}")
    print(f"  #MEASURE columnas en panel: {panel_u.shape[1] - len(base_index_cols)}")
    print('-' * 40)

# Atajos convenientes
panel_xdc = panels['XDC']
panel_xdc_ps = panels['XDC_PS']
panel_usd_ppp = panels['USD_PPP']
panel_usd_ppp_ps = panels['USD_PPP_PS']

print('Listo: se generaron 4 paneles anchos (uno por UNIT_MEASURE).')

UNIT_MEASURE=XDC
  Long shape:  (70380, 14)
  Panel shape: (4140, 22)
  #MEASURE columnas en panel: 16
----------------------------------------
UNIT_MEASURE=XDC_PS
  Long shape:  (70380, 14)
  Panel shape: (4140, 22)
  #MEASURE columnas en panel: 16
----------------------------------------
UNIT_MEASURE=USD_PPP
  Long shape:  (70380, 14)
  Panel shape: (4140, 22)
  #MEASURE columnas en panel: 16
----------------------------------------
UNIT_MEASURE=USD_PPP_PS
  Long shape:  (70380, 14)
  Panel shape: (4140, 22)
  #MEASURE columnas en panel: 16
----------------------------------------
Listo: se generaron 4 paneles anchos (uno por UNIT_MEASURE).


In [8]:
core_measures = ['E1', 'R1']  # mínimo para balance fiscal
years_expected = list(range(2010, 2022))  # 2010-2021
n_years_expected = len(years_expected)

# 1) Cobertura de años por municipio
years_by_muni = panel_xdc.groupby('REF_AREA')['TIME_PERIOD'].nunique().sort_values()
print("Resumen #años por municipio (panel_xdc):")
display(years_by_muni.describe())

# Municipios con cobertura completa de años
muni_full_years = years_by_muni[years_by_muni == n_years_expected].index
print("Municipios con 2010-2021 completo:", len(muni_full_years), "de", years_by_muni.shape[0])

# 2) Cobertura de medidas core en el panel (a nivel de celda faltante)
missing_core = panel_xdc[core_measures].isna().sum().to_frame('missing_cells')
missing_core['missing_pct'] = missing_core['missing_cells'] / len(panel_xdc)
print("Nulos en medidas core (panel_xdc):")
display(missing_core)

# 3) Definir panel balanceado: full years + medidas core completas
panel_xdc_full_years = panel_xdc[panel_xdc['REF_AREA'].isin(muni_full_years)].copy()

mask_core_complete = panel_xdc_full_years[core_measures].notna().all(axis=1)
panel_xdc_balanced = panel_xdc_full_years[mask_core_complete].copy()

print("Shapes:")
print("panel_xdc:", panel_xdc.shape)
print("panel_xdc_full_years:", panel_xdc_full_years.shape)
print("panel_xdc_balanced:", panel_xdc_balanced.shape)

Resumen #años por municipio (panel_xdc):


count    345.0
mean      12.0
std        0.0
min       12.0
25%       12.0
50%       12.0
75%       12.0
max       12.0
Name: TIME_PERIOD, dtype: float64

Municipios con 2010-2021 completo: 345 de 345
Nulos en medidas core (panel_xdc):


,missing_cells,missing_pct
MEASURE,,
E1,0,0.0
R1,0,0.0


Shapes:
panel_xdc: (4140, 22)
panel_xdc_full_years: (4140, 22)
panel_xdc_balanced: (4140, 22)


In [9]:
# Conteo general de negativos (en el raw)
neg_mask = df['OBS_VALUE'].notna() & (df['OBS_VALUE'] < 0)
print("Negativos (raw df):", int(neg_mask.sum()), "de", int(df['OBS_VALUE'].notna().sum()))

# Negativos por unidad
neg_by_unit = (
    df.loc[neg_mask]
      .groupby('UNIT_MEASURE')
      .size()
      .sort_values(ascending=False)
      .to_frame('neg_rows')
)
display(neg_by_unit)

# Negativos por medida
neg_by_measure = (
    df.loc[neg_mask]
      .groupby('MEASURE')
      .size()
      .sort_values(ascending=False)
      .to_frame('neg_rows')
)
display(neg_by_measure)

# Cruce unidad x medida (top 20 combinaciones con más negativos)
neg_unit_measure = (
    df.loc[neg_mask]
      .groupby(['UNIT_MEASURE','MEASURE'])
      .size()
      .sort_values(ascending=False)
      .head(20)
      .to_frame('neg_rows')
)
display(neg_unit_measure)

Negativos (raw df): 80 de 269100


,neg_rows
UNIT_MEASURE,
USD_PPP,20
USD_PPP_PS,20
XDC,20
XDC_PS,20


,neg_rows
MEASURE,
R14,80


,,neg_rows
UNIT_MEASURE,MEASURE,
USD_PPP,R14,20
USD_PPP_PS,R14,20
XDC,R14,20
XDC_PS,R14,20


In [10]:
for u, p in panels.items():
    p['flag_r14_negative'] = p['R14'].notna() & (p['R14'] < 0)

    print(f"UNIT_MEASURE={u}")
    print("  Filas con R14 negativo:", int(p['flag_r14_negative'].sum()))
    print('-' * 40)

panel_xdc = panels['XDC']
panel_xdc_ps = panels['XDC_PS']
panel_usd_ppp = panels['USD_PPP']
panel_usd_ppp_ps = panels['USD_PPP_PS']

print('Listo: flag creado (flag_r14_negative) en los 4 paneles.')

UNIT_MEASURE=XDC
  Filas con R14 negativo: 20
----------------------------------------
UNIT_MEASURE=XDC_PS
  Filas con R14 negativo: 20
----------------------------------------
UNIT_MEASURE=USD_PPP
  Filas con R14 negativo: 20
----------------------------------------
UNIT_MEASURE=USD_PPP_PS
  Filas con R14 negativo: 20
----------------------------------------
Listo: flag creado (flag_r14_negative) en los 4 paneles.


In [11]:
# Paso 4: Validaciones jerárquicas (contables) en el panel XDC
# Idea: verificar consistencia entre totales y componentes (cuando corresponda)

p = panel_xdc.copy()

# Tolerancia relativa para considerar que "cierra" (ej. 1%)
tol = 0.01

# Helpers para evitar divisiones por cero
safe_denom_e1 = p['E1'].replace({0: np.nan})
safe_denom_r1 = p['R1'].replace({0: np.nan})

# Expenditure: E1 ?= E11 + E12
p['E1_components_sum'] = p['E11'] + p['E12']
p['E1_diff'] = p['E1'] - p['E1_components_sum']
p['E1_rel_diff'] = (p['E1_diff'].abs() / safe_denom_e1)

# Current expenditure: E11 ?= E111 + E115
p['E11_components_sum'] = p['E111'] + p['E115']
p['E11_diff'] = p['E11'] - p['E11_components_sum']
p['E11_rel_diff'] = (p['E11_diff'].abs() / p['E11'].replace({0: np.nan}))

# Revenue: R1 ?= R11 + R12 + R13 + R14
p['R1_components_sum'] = p['R11'] + p['R12'] + p['R13'] + p['R14']
p['R1_diff'] = p['R1'] - p['R1_components_sum']
p['R1_rel_diff'] = (p['R1_diff'].abs() / safe_denom_r1)

# Flags de cierre
p['flag_E1_close'] = p['E1_rel_diff'].le(tol) | p['E1_rel_diff'].isna()
p['flag_E11_close'] = p['E11_rel_diff'].le(tol) | p['E11_rel_diff'].isna()
p['flag_R1_close'] = p['R1_rel_diff'].le(tol) | p['R1_rel_diff'].isna()

# Resumen
summary = pd.DataFrame({
    'check': ['E1 vs (E11+E12)', 'E11 vs (E111+E115)', 'R1 vs (R11+R12+R13+R14)'],
    'tol_rel': [tol, tol, tol],
    'not_close_rows': [
        int((~p['flag_E1_close']).sum()),
        int((~p['flag_E11_close']).sum()),
        int((~p['flag_R1_close']).sum()),
    ],
    'not_close_pct': [
        float((~p['flag_E1_close']).mean()),
        float((~p['flag_E11_close']).mean()),
        float((~p['flag_R1_close']).mean()),
    ],
}).sort_values('not_close_rows', ascending=False)

display(summary)

# Mostrar ejemplos (top 10) donde peor cierra cada check
cols_id = ['REF_AREA','Municipality','TIME_PERIOD']

display(
    p.loc[~p['flag_E1_close'], cols_id + ['E1','E11','E12','E1_components_sum','E1_diff','E1_rel_diff']]
     .sort_values('E1_rel_diff', ascending=False)
     .head(10)
)

display(
    p.loc[~p['flag_E11_close'], cols_id + ['E11','E111','E115','E11_components_sum','E11_diff','E11_rel_diff']]
     .sort_values('E11_rel_diff', ascending=False)
     .head(10)
)

display(
    p.loc[~p['flag_R1_close'], cols_id + ['R1','R11','R12','R13','R14','R1_components_sum','R1_diff','R1_rel_diff']]
     .sort_values('R1_rel_diff', ascending=False)
     .head(10)
)

# Guardar p para inspección posterior
validation_xdc = p
print('Listo: validaciones jerárquicas calculadas en validation_xdc')

,check,tol_rel,not_close_rows,not_close_pct
0,E1 vs (E11+E12),0.01,0,0.0
1,E11 vs (E111+E115),0.01,0,0.0
2,R1 vs (R11+R12+R13+R14),0.01,0,0.0


MEASURE,REF_AREA,Municipality,TIME_PERIOD,E1,E11,E12,E1_components_sum,E1_diff,E1_rel_diff


MEASURE,REF_AREA,Municipality,TIME_PERIOD,E11,E111,E115,E11_components_sum,E11_diff,E11_rel_diff


MEASURE,REF_AREA,Municipality,TIME_PERIOD,R1,R11,R12,R13,R14,R1_components_sum,R1_diff,R1_rel_diff


Listo: validaciones jerárquicas calculadas en validation_xdc


In [14]:
import os

# Paso 5: Features base + export de paneles por unidad

def add_base_features(panel: pd.DataFrame) -> pd.DataFrame:
    p = panel.copy()

    # Helpers para divisiones seguras
    denom_e1 = p['E1'].replace({0: np.nan})
    denom_r1 = p['R1'].replace({0: np.nan})
    denom_e11 = p['E11'].replace({0: np.nan})
    denom_e12 = p['E12'].replace({0: np.nan})

    # Nivel fiscal básico
    p['balance'] = p['R1'] - p['E1']
    p['balance_ratio'] = p['balance'] / denom_r1

    # Composición del gasto
    p['current_exp_ratio'] = p['E11'] / denom_e1
    p['capital_exp_ratio'] = p['E12'] / denom_e1
    p['comp_employees_ratio'] = p['E111'] / denom_e1
    p['other_current_ratio'] = p['E115'] / denom_e1
    p['direct_investment_ratio'] = p['E121'] / denom_e12

    # Composición de ingresos
    p['tax_ratio'] = p['R11'] / denom_r1
    p['grants_ratio'] = p['R12'] / denom_r1
    p['fees_ratio'] = p['R13'] / denom_r1
    p['assets_income_ratio'] = p['R14'] / denom_r1

    # Flags QA ya calculados previamente (si existen)
    if 'flag_r14_negative' not in p.columns:
        p['flag_r14_negative'] = p['R14'].notna() & (p['R14'] < 0)

    return p

processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

panels_features = {}

for u, panel_u in panels.items():
    panel_feat = add_base_features(panel_u)
    panel_feat['UNIT_MEASURE'] = u

    panels_features[u] = panel_feat

    out_path = f"{processed_dir}/chile_panel_{u}_features.csv"
    panel_feat.to_csv(out_path, index=False)

    print(f"Exportado: {out_path}")
    print(f"  Shape: {panel_feat.shape}")
    print('-' * 40)

# Atajos
panel_xdc_feat = panels_features['XDC']
panel_xdc_ps_feat = panels_features['XDC_PS']
panel_usd_ppp_feat = panels_features['USD_PPP']
panel_usd_ppp_ps_feat = panels_features['USD_PPP_PS']

print('Listo: paneles con features creados y exportados (por unidad).')

Exportado: ../data/processed/chile_panel_XDC_features.csv
  Shape: (4140, 35)
----------------------------------------
Exportado: ../data/processed/chile_panel_XDC_PS_features.csv
  Shape: (4140, 35)
----------------------------------------
Exportado: ../data/processed/chile_panel_USD_PPP_features.csv
  Shape: (4140, 35)
----------------------------------------
Exportado: ../data/processed/chile_panel_USD_PPP_PS_features.csv
  Shape: (4140, 35)
----------------------------------------
Listo: paneles con features creados y exportados (por unidad).
